# 01 — Data Preparation

Minari/D4RL Ant dataset에서 trajectory를 다운로드하고, Hilbert transform 기반 phase label과 quality filter를 적용해 `demos_ant.npz`를 생성한 뒤, train/val split과 normalization stats를 `norm_stats.npz`로 저장합니다.

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `src/data_extraction.py`, `src/data_pipeline.py`, artifact 경로 관리는 `src/paths.py`에 있습니다.

Colab에서 MuJoCo 렌더링이 필요하면 README의 "설치" 섹션에서 시스템 패키지 설치 안내를 참고하세요.

## 1. Colab/local bootstrap


In [ ]:
# Common Colab/local bootstrap: locate the repo root and install from there.
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = 'phase_conditioned_diffusion_policy'
COLAB_REPO_ROOT = Path('/content') / REPO_NAME
# If this notebook is opened in a fresh Colab runtime without running 00_colab_setup.ipynb,
# set PCDP_REPO_URL to your GitHub clone URL before this cell executes.
DEFAULT_REPO_URL = os.environ.get(
    'PCDP_REPO_URL',
    'https://github.com/YOUR_GITHUB_USERNAME/phase_conditioned_diffusion_policy.git',
)

def in_colab() -> bool:
    return 'google.colab' in sys.modules or Path('/content').exists() and 'COLAB_RELEASE_TAG' in os.environ

def find_repo_root(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'pcdp').is_dir():
            return path
    if COLAB_REPO_ROOT.exists():
        return COLAB_REPO_ROOT
    return None

repo_root = find_repo_root()
if repo_root is None and in_colab():
    if 'YOUR_GITHUB_USERNAME' in DEFAULT_REPO_URL:
        raise RuntimeError(
            'Fresh Colab runtime detected but /content/phase_conditioned_diffusion_policy is missing. '
            'Run notebooks/00_colab_setup.ipynb first, or set the PCDP_REPO_URL environment '
            "variable to this repo\'s GitHub clone URL before running this cell."
        )
    subprocess.run(['git', 'clone', DEFAULT_REPO_URL, str(COLAB_REPO_ROOT)], check=True)
    repo_root = COLAB_REPO_ROOT

if repo_root is None:
    raise RuntimeError('Could not locate the repository root. Start Jupyter from the repo or run 00_colab_setup.ipynb in Colab.')

os.chdir(repo_root)
try:
    get_ipython().run_line_magic('cd', str(repo_root))
except NameError:
    pass

os.environ.setdefault('MUJOCO_GL', 'osmesa')
os.environ.setdefault('PYOPENGL_PLATFORM', 'osmesa')
if 'PCDP_ARTIFACT_ROOT' not in os.environ:
    default_artifact_root = Path('/content/pcdp_artifacts') if in_colab() else Path.home() / 'phase_conditioned_diffusion_policy_artifacts'
    os.environ['PCDP_ARTIFACT_ROOT'] = str(default_artifact_root)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=repo_root, check=True)
print(f'✓ repo root: {repo_root}')
print('✓ pcdp installed from repo root (editable)')
print(f'✓ artifact root: {os.environ["PCDP_ARTIFACT_ROOT"]}')


## 2. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 3. Dataset discovery and load

In [ ]:
import minari
import numpy as np

from pcdp.data_extraction import (
    ANT_PHASE_JOINT_IDX,
    DEFAULT_MINARI_ANT_DATASET,
    DemoExtractionConfig,
    extract_demos_from_episodes,
    load_ant_dataset,
    materialize_episodes,
    plot_demo_quality,
    print_demo_quality_report,
    save_demos,
)

print(f'Minari version: {minari.__version__}')
dataset = load_ant_dataset(minari, DEFAULT_MINARI_ANT_DATASET)


## 4. Episode diagnostics

In [ ]:
config = DemoExtractionConfig()
episodes = materialize_episodes(dataset, expected_obs_dim=config.expected_obs_dim)
print(f'Phase 라벨링용 joint: obs[{ANT_PHASE_JOINT_IDX}]')


## 5. Demo extraction and `demos_ant.npz` save

In [ ]:
demos = extract_demos_from_episodes(episodes, config)
assert demos is not None, '선택된 demos가 없습니다. DemoExtractionConfig 임계값을 완화해 주세요.'
demos_path = save_demos(demos, DATA_DIR / 'demos_ant.npz')


## 6. Quality plots and report

In [ ]:
demos_loaded = np.load(demos_path)
quality_path, visualization_path = plot_demo_quality(demos_loaded, FIGURES_DIR)
print_demo_quality_report(demos_loaded)


## 7. Data pipeline — split + normalization

Phase 1에서 만든 `demos_ant.npz`를 episode-level train/val split하고, train-only normalization stats와 horizon/frequency 메타데이터를 `norm_stats.npz`로 저장합니다.

In [ ]:
from pcdp.data_pipeline import DataPipelineConfig, plot_phase_advance, run_data_pipeline

DATA_PATH = DATA_DIR / 'demos_ant.npz'
NORM_PATH = DATA_DIR / 'norm_stats.npz'
assert DATA_PATH.exists(), f'파일 없음: {DATA_PATH}'

pipeline_cfg = DataPipelineConfig(
    obs_horizon=2,
    pred_horizon=16,
    action_horizon=8,
    val_ratio=0.15,
    batch_size=256,
    num_workers=2,
    seed=42,
)
pipeline = run_data_pipeline(DATA_PATH, NORM_PATH, pipeline_cfg)
train_dataset = pipeline['train_dataset']
val_dataset = pipeline['val_dataset']
train_loader = pipeline['train_loader']
val_loader = pipeline['val_loader']


## 7. Phase-advance diagnostic

In [ ]:
phase_advance = plot_phase_advance(
    train_dataset,
    FIGURES_DIR,
    f_mean=pipeline['project_data']['freq_window_mean'],
    seed=pipeline_cfg.seed,
)
print('\n=== 01 Data Preparation 완료 ===')
print(f'Demos:        {demos_path}')
print(f'Norm stats:   {NORM_PATH}')
print(f'Train chunks: {len(train_dataset)}')
print(f'Val chunks:   {len(val_dataset)}')
print(f'Steps/epoch:  {len(train_loader)}')
print(f'Phase plot:   {phase_advance["path"]}')
